# 01 — Data loading and preprocessing

This notebook loads the raw TCGA-BRCA cBioPortal files, inspects the available clinical and molecular variables, harmonizes sample identifiers and saves clean processed tables for downstream analysis.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

# Allow imports from src/
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_utils import (
    read_cbioportal_table,
    clean_expression_matrix,
    harmonize_tcga_sample_id,
    harmonize_tcga_patient_id,
    find_columns_containing,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

ModuleNotFoundError: No module named 'pandas'

In [ ]:
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "cbioportal_brca"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

clinical_patient_path = RAW_DIR / "data_clinical_patient.txt"
clinical_sample_path = RAW_DIR / "data_clinical_sample.txt"
expression_path = RAW_DIR / "data_mrna_seq_v2_rsem.txt"

for path in [clinical_patient_path, clinical_sample_path, expression_path]:
    print(path, "exists:", path.exists())

In [ ]:
clinical_patient = read_cbioportal_table(clinical_patient_path)
clinical_sample = read_cbioportal_table(clinical_sample_path)
expression_raw = read_cbioportal_table(expression_path)

print("clinical_patient:", clinical_patient.shape)
print("clinical_sample:", clinical_sample.shape)
print("expression_raw:", expression_raw.shape)

In [ ]:
print("Clinical patient columns:")
display(pd.DataFrame({"column": clinical_patient.columns}))

print("Clinical sample columns:")
display(pd.DataFrame({"column": clinical_sample.columns}))

In [ ]:
keywords = [
    "SUBTYPE",
    "PAM50",
    "ER",
    "PR",
    "HER2",
    "STATUS",
    "SURVIVAL",
    "OS",
    "DFS",
    "TUMOR",
    "STAGE",
    "GRADE",
    "AGE",
]

patient_matches = find_columns_containing(clinical_patient, keywords)
sample_matches = find_columns_containing(clinical_sample, keywords)

print("Potentially useful patient-level columns:")
display(pd.DataFrame({"column": patient_matches}))

print("Potentially useful sample-level columns:")
display(pd.DataFrame({"column": sample_matches}))

In [ ]:
expression = clean_expression_matrix(expression_raw)

print("Clean expression matrix:")
print(expression.shape)
display(expression.iloc[:5, :5])

In [ ]:
clinical_sample = clinical_sample.copy()
clinical_patient = clinical_patient.copy()
expression = expression.copy()

# Standard cBioPortal columns are often SAMPLE_ID and PATIENT_ID
print("clinical_sample ID columns:")
display(clinical_sample[[col for col in clinical_sample.columns if "ID" in col.upper()]].head())

print("clinical_patient ID columns:")
display(clinical_patient[[col for col in clinical_patient.columns if "ID" in col.upper()]].head())

In [ ]:
expression = expression.reset_index()

expression["SAMPLE_ID_ORIGINAL"] = expression["SAMPLE_ID"]
expression["SAMPLE_ID_15"] = expression["SAMPLE_ID_ORIGINAL"].apply(lambda x: harmonize_tcga_sample_id(x, n=15))
expression["PATIENT_ID_12"] = expression["SAMPLE_ID_ORIGINAL"].apply(harmonize_tcga_patient_id)

display(expression[["SAMPLE_ID_ORIGINAL", "SAMPLE_ID_15", "PATIENT_ID_12"]].head())

In [ ]:
clinical_sample["SAMPLE_ID_ORIGINAL"] = clinical_sample["SAMPLE_ID"]
clinical_sample["SAMPLE_ID_15"] = clinical_sample["SAMPLE_ID"].apply(lambda x: harmonize_tcga_sample_id(x, n=15))

if "PATIENT_ID" in clinical_sample.columns:
    clinical_sample["PATIENT_ID_12"] = clinical_sample["PATIENT_ID"].apply(harmonize_tcga_patient_id)
else:
    clinical_sample["PATIENT_ID_12"] = clinical_sample["SAMPLE_ID"].apply(harmonize_tcga_patient_id)

display(clinical_sample[["SAMPLE_ID_ORIGINAL", "SAMPLE_ID_15", "PATIENT_ID_12"]].head())

In [ ]:
if "PATIENT_ID" not in clinical_patient.columns:
    raise ValueError("Expected PATIENT_ID column not found in clinical_patient.")

clinical_patient["PATIENT_ID_12"] = clinical_patient["PATIENT_ID"].apply(harmonize_tcga_patient_id)

display(clinical_patient[["PATIENT_ID", "PATIENT_ID_12"]].head())

In [ ]:
expression_samples = set(expression["SAMPLE_ID_15"])
clinical_samples = set(clinical_sample["SAMPLE_ID_15"])

sample_overlap = expression_samples.intersection(clinical_samples)

print("Expression samples:", len(expression_samples))
print("Clinical samples:", len(clinical_samples))
print("Overlap:", len(sample_overlap))
print("Expression only:", len(expression_samples - clinical_samples))
print("Clinical only:", len(clinical_samples - expression_samples))

In [ ]:
# Merge sample-level clinical metadata
clinical_merged = clinical_sample.merge(
    clinical_patient,
    on="PATIENT_ID_12",
    how="left",
    suffixes=("_sample", "_patient")
)

print("clinical_merged:", clinical_merged.shape)
display(clinical_merged.head())

In [ ]:
metadata_expression = clinical_merged.merge(
    expression[["SAMPLE_ID_ORIGINAL", "SAMPLE_ID_15", "PATIENT_ID_12"]],
    on=["SAMPLE_ID_15", "PATIENT_ID_12"],
    how="inner"
)

print("metadata_expression:", metadata_expression.shape)
display(metadata_expression.head())

In [ ]:
gene_columns = [
    col for col in expression.columns
    if col not in ["SAMPLE_ID", "SAMPLE_ID_ORIGINAL", "SAMPLE_ID_15", "PATIENT_ID_12"]
]

expression_clean = expression[["SAMPLE_ID_ORIGINAL", "SAMPLE_ID_15", "PATIENT_ID_12"] + gene_columns].copy()

print("expression_clean:", expression_clean.shape)
display(expression_clean.iloc[:5, :8])

In [ ]:
marker_genes = [
    "ESR1",
    "PGR",
    "ERBB2",
    "MKI67",
    "FOXA1",
    "GATA3",
    "TP53",
    "PIK3CA",
    "BRCA1",
    "BRCA2",
]

available_markers = [gene for gene in marker_genes if gene in expression_clean.columns]
missing_markers = [gene for gene in marker_genes if gene not in expression_clean.columns]

print("Available markers:", available_markers)
print("Missing markers:", missing_markers)

marker_expression = expression_clean[
    ["SAMPLE_ID_ORIGINAL", "SAMPLE_ID_15", "PATIENT_ID_12"] + available_markers
].copy()

display(marker_expression.head())

In [ ]:
clinical_patient.to_csv(PROCESSED_DIR / "clinical_patient.tsv", sep="\t", index=False)
clinical_sample.to_csv(PROCESSED_DIR / "clinical_sample.tsv", sep="\t", index=False)
clinical_merged.to_csv(PROCESSED_DIR / "clinical_merged.tsv", sep="\t", index=False)
metadata_expression.to_csv(PROCESSED_DIR / "metadata_expression_samples.tsv", sep="\t", index=False)
expression_clean.to_csv(PROCESSED_DIR / "expression_rsem_samples_by_gene.tsv", sep="\t", index=False)
marker_expression.to_csv(PROCESSED_DIR / "marker_expression.tsv", sep="\t", index=False)

print("Saved processed files to:", PROCESSED_DIR)